# 03 — Multivariate bias adjustment

This notebook applies two multivariate bias-adjustment methods, MBCr and MBCn, to daily maximum temperature and humidity using two-fold cross-validation.

Two indirect configurations are evaluated:

1. **Indirect-SH:** joint adjustment of daily maximum temperature (`tasmax`) and daily specific humidity (`huss`).
2. **Indirect-RH:** joint adjustment of daily maximum temperature (`tasmax`) and estimated daily minimum relative humidity (`hurs`).

After adjustment, the corrected variables can be used to calculate WBT and sWBGT. The example uses REMO2015 driven by NorESM1-M over SESA.

In [1]:
options(java.parameters = "-Xmx32g")

rm(list = ls())
graphics.off()
invisible(gc())

library(loadeR)
library(visualizeR)
library(transformeR)
library(downscaleR)
library(MBC)
library(HeatStress)
library(magrittr)
library(climate4R.indices)


Loading required package: rJava

Loading required package: loadeR.java

Java version 11x amd64 by Oracle Corporation detected

NetCDF Java Library v4.6.0-SNAPSHOT (23 Apr 2015) loaded and ready

Loading required package: climate4R.UDG

climate4R.UDG version 0.2.3 (2021-07-05) is loaded


Get the latest stable version (0.2.4) using <devtools::install_github('SantanderMetGroup/climate4R.UDG')>

Please use 'citation("climate4R.UDG")' to cite this package.

loadeR version 1.7.1 (2021-07-05) is loaded


Get the latest stable version (1.8.2) using <devtools::install_github(c('SantanderMetGroup/climate4R.UDG','SantanderMetGroup/loadeR'))>

Please use 'citation("loadeR")' to cite this package.

Loading required package: transformeR

Warning message:
“replacing previous import ‘lifecycle::last_warnings’ by ‘rlang::last_warnings’ when loading ‘tibble’”



    _______   ____  ___________________  __  ________ 
   / ___/ /  / /  |/  / __  /_  __/ __/ / / / / __  / 
  / /  / /  / / /|_/ / /_/ / / / / __/ / /_/ / /_/_/  
 / /__/ /__/ / /  / / __  / / / / /__ /___  / / \ \ 
 \___/____/_/_/  /_/_/ /_/ /_/  \___/    /_/\/   \_\ 
 
      github.com/SantanderMetGroup/climate4R



transformeR version 2.1.2 (2021-07-07) is loaded


Get the latest stable version (2.2.5) using <devtools::install_github('SantanderMetGroup/transformeR')>

Please see 'citation("transformeR")' to cite this package.

Warning message:
“no DISPLAY variable so Tk is not available”
visualizeR version 1.6.4 (2023-10-26) is loaded

Please see 'citation("visualizeR")' to cite this package.

downscaleR version 3.3.2 (2020-06-05) is loaded


Get the latest stable version (3.3.4) using <devtools::install_github(c('SantanderMetGroup/transformeR','SantanderMetGroup/downscaleR'))>

Please use 'citation("downscaleR")' to cite this package.

Loading required package: Matrix

Loading required package: energy

Loading required package: FNN

HeatStress version 1.0.8.1 (2025-08-13) is loaded

Use 'indexShow()' for an overview of the available heat indices

climate4R.indices version 0.3.1 (2023-06-22) is loaded


Get the latest stable version (0.3.2) using <devtools::install_github('SantanderMetGroup/clima

##  Setup

In [2]:
# Analysis configuration
years <- 1982:2005
season <- c(12, 1, 2)
folds <- list(1982:1993, 1994:2005)
methods <- c("mbcr", "mbcn")
humidity_inputs <- c("huss", "hurs")
experiment <- "CrosVal"

gcm_id <- "NorESM1"
rcm_id <- "REMO"

# MBC configurations
ratio.seq <- c(FALSE, FALSE)
trace <- c(Inf, 0)
pp.type <- 6

# Input and output paths
data_dir <- "prepared_data"
output_dir <- "bias_adjusted/multivariate"
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

obs_tasmax_file <- file.path(data_dir, "Data_CV_BA_tasmax_MSWX.rda")
obs_huss_file <- file.path(data_dir, "Data_CV_BA_huss_MSWX.rda")
model_tasmax_file <- file.path(data_dir, "Data_tasmax_REMO2015_NCC-NorESM1-M.rda")
model_huss_file <- file.path(data_dir, "Data_huss_REMO2015_NCC-NorESM1-M.rda")

##  Helper functions

In [13]:
calc_RH <- function(tas_grid, hus_grid) {
  rh <- huss2hurs(
    huss = as.vector(hus_grid[["Data"]]),
    tas = as.vector(tas_grid[["Data"]]),
    ps = 1013
  )

  out <- hus_grid
  out[["Data"]] <- array(rh, dim = dim(hus_grid[["Data"]]))
  attributes(out[["Data"]]) <- attributes(tas_grid[["Data"]])
  out[["Variable"]][["varName"]] <- "hurs"
  out
}

output_name <- function(humidity, method) {
  file.path(
    output_dir,
    paste(humidity, rcm_id, method, gcm_id, experiment,
          "indirect_SESA.rda", sep = "_")
  )
}

## Estimated wbt from tasmax y hus/huss
calc_wbt <- function(tas_grid, hus_grid) {
  HR <- huss2hurs(huss = as.vector(hus_grid[["Data"]]),
                  tas  = as.vector(tas_grid[["Data"]]), ps = 1013)
  sw <- wbt.Stull(as.vector(tas_grid[["Data"]]), hurs = HR)
  hus_grid[["Data"]] <- array(sw, dim = dim(hus_grid[["Data"]]))
  attributes(hus_grid[["Data"]])=attributes(tas_grid[["Data"]])
  hus_grid
}

## Estimated wbt from tasmax and relative humidity
calc_wbt_hurs <- function(tas_grid, hurs_grid) {
  sw <- wbt.Stull(as.vector(tas_grid[["Data"]]),
                  hurs = as.vector(hurs_grid[["Data"]]))
  out <- tas_grid
  out[["Data"]] <- array(sw, dim = dim(tas_grid[["Data"]]))
  attributes(out[["Data"]]) <- attributes(tas_grid[["Data"]])
  out
}

##  Diagnostic functions

In [15]:
aggregate_djf_Ind27 <- function(grid) {
  
  # Define the ocean mask from the original daily data
  ocean_mask <- apply(
    is.na(grid[["Data"]]),
    MARGIN = 2:length(dim(grid[["Data"]])),
    FUN = all
  )
  
  #  Calculate the number of days with WBT >= 27 °C
  ind <- indexGrid(
    tx = grid,
    time.resolution = "year",
    index.code = "TXth",
    th = 27
  )
  
  #  Repeat the spatial mask for every year
  ocean_mask_array <- array(
    rep(ocean_mask, each = dim(ind[["Data"]])[1]),
    dim = dim(ind[["Data"]])
  )
  
  # Apply the mask to the final result
  ind[["Data"]][ocean_mask_array] <- NA_real_
  
  # 4. Convert to percentage
  ind <- gridArithmetics(ind, 90, operator = "/")
  ind <- gridArithmetics(ind, 100, operator = "*")
  
  # Replace non-finite values (Inf, -Inf) with 0
  ind[["Data"]][!is.finite(ind[["Data"]]) & !is.na(ind[["Data"]])] <- 0

  return(ind)
}

### Cross-validation 

The 24 austral summers are divided into two 12-year folds: 1982–1993 and 1994–2005. In each iteration, one fold is used for calibration and the other for validation. 

## Multivariate adjustment under indirect approach
Each grid cell is adjusted independently using Cannon package. As an example method is selected to MBCr

In [16]:
set.seed(123)
#For simplicy only one method and huss
methods_to_run='mbcr'
inputs_selected= humidity_inputs[1]

mbc_results <- setNames(vector("list", length(humidity_inputs)),
                        humidity_inputs)

In [ ]:
for (input in inputs_selected){

message(paste0('input ',input))


obs.huss=get(load(obs_huss_file)) %>% subsetGrid(years = 1982:2005)
obs.Tx=get(load(obs_tasmax_file))%>% subsetGrid(years = 1982:2005)

obs.hurs=calc_RH(tas_grid=obs.Tx,hus_grid=obs.huss)
  
message(paste0('GCMs ',gcm_id))
message(paste0('RCMs ',rcm_id))

Tx <-get(load(model_tasmax_file))
huss <-get(load(model_huss_file))
hurs=calc_RH(tas_grid=Tx,hus_grid=huss)


Tx=getTemporalIntersection(obs =obs.Tx,prd=Tx,which.return = 'prd')
huss=getTemporalIntersection(obs =obs.huss,prd=huss,which.return = 'prd')
hurs=getTemporalIntersection(obs =obs.hurs,prd=hurs,which.return = 'prd')

obs.Tx_1=getTemporalIntersection(obs =obs.Tx,prd=Tx,which.return = 'obs')
obs.huss_1=getTemporalIntersection(obs =obs.huss,prd=huss,which.return = 'obs')
obs.hurs_1=getTemporalIntersection(obs =obs.hurs,prd=hurs,which.return = 'obs')

if (input=='huss') {
  y.obs=list('Tx'=obs.Tx_1,'huss'=obs.huss_1)
  y.obs[["Tx"]][["Variable"]][["varName"]]='tasmax'
  y.obs[["huss"]][["Variable"]][["varName"]]='huss'
} 

if (input=='hurs') {
  y.obs=list('Tx'=obs.Tx_1,'hurs'=obs.hurs_1)
  y.obs[["Tx"]][["Variable"]][["varName"]]='tasmax'
  y.obs[["hurs"]][["Variable"]][["varName"]]='hurs'
}

if(input=='huss') {
x=list(Tx,huss)
} else {
x=list(Tx,hurs)
}

years <- folds
y=y.obs

for (method in methods_to_run) {
    message("Method: ", toupper(method))

ls <- lapply(1:length(years), function(k) {
  target.year <- years[[k]]
  rest.years <- setdiff(unlist(years), target.year)
  
  # yy <- lapply(y, function(k) redim(k, member = FALSE))
  yy <- lapply(y, function(yy.i) subsetGrid(yy.i, years = rest.years, drop = FALSE))
  yy <- lapply(yy, function(yy.i) redim(yy.i, drop = TRUE))
  
  newdata2 <- lapply(x, function(k) subsetGrid(k, years = target.year, drop = F))
  xx <- lapply(x, function(k) subsetGrid(k, years = rest.years, drop = F))
  
  oo <- redim(makeMultiGrid(yy[[1]],yy[[2]]), drop=TRUE) 
  pp <- redim(makeMultiGrid(xx[[1]],xx[[2]]), drop=TRUE) 
  ss<- redim(makeMultiGrid(newdata2[[1]],newdata2[[2]]), drop=TRUE)
  
  message("Validation ", k, ", ", length(unique(years)) - k, " remaining")
  #apply MBC correction
  newdata.mbc <- array(NA,dim=dim(ss[["Data"]]))
  
  for(i in 1:length(getCoordinates(newdata2[[1]])[["y"]])){
    for(j in 1:length(getCoordinates(newdata2[[1]])[["x"]])){
     # message("Validation point ", i, ", ",j )
      if(all(is.na(oo[["Data"]][,,i,j])) | all(is.na(pp[["Data"]][,,i,j]))){
        next
      } 
      
      o <- t(oo[["Data"]][,,i,j])
      p <- t(pp[["Data"]][,,i,j])
      s <- t(ss[["Data"]][,,i,j])
      
    if(method=='mbcr'){
      invisible(capture.output(corrected.mbc <- MBCr(o.c=o, m.c=p,
                      m.p=s, ratio.seq=ratio.seq, trace=trace,pp.type=pp.type))) 
                      #avoid printing to console
}
    if(method=='mbcn'){
     invisible(capture.output( corrected.mbc <- MBCn(o.c=o, m.c=p,
                      m.p=s, ratio.seq=ratio.seq, trace=trace,pp.type=pp.type)))
                      #avoid printing to console
    }  
      
      newdata.mbc[,,i,j]<-t(corrected.mbc[["mhat.p"]])
    }
  }
  
  folds.mbc <- ss
  folds.mbc[["Data"]] <- newdata.mbc
  attr(folds.mbc[["Data"]],"dimensions")<- c("var","time" ,"lat", "lon")
  return(folds.mbc)

})
mbc.all <- bindGrid(ls, dimension = "time") 

mbc_results[[input]][[method]] <- mbc.all
save(mbc.all, file = output_name(input, method))
invisible(gc())
}
}

input huss

GCMs NorESM1

RCMs REMO

Method: MBCR

Validation 1, 1 remaining

Validation 2, 0 remaining



## Diagnostic 
As an example calculate the WBT27 index for the bias-adjusted WBT data from the indirect approach using MBCr and compared to the raw simulations and the reference. 

In [25]:
# Calculate wbt from the independently adjusted input variables

wbt_indirect_SH <- setNames(vector("list", length(methods_to_run)),
                             methods_to_run)
wbt_indirect_RH <- setNames(vector("list", length(methods_to_run)),
                             methods_to_run)


for (method in methods_to_run) {

if(inputs_selected=='huss'){
  data=mbc_results[["huss"]][[method]]
  wbt_indirect_SH[[method]] <- calc_wbt(
    tas_grid = subsetGrid(data,var=data[["Variable"]][["varName"]][1]), 
    hus_grid = subsetGrid(data,var=data[["Variable"]][["varName"]][2]))
}
if (inputs_selected=='hurs'){
  data=mbc_results[[hurs]][[method]]

  wbt_indirect_RH[[method]] <- calc_wbt_hurs(
    tas_grid = subsetGrid(data,var=data[["Variable"]][["varName"]][1]),
     hurs_grid = subsetGrid(data,var=data[["Variable"]][["varName"]][2]))
}
}

In [27]:
# WBT27 bias ------------------------------------------------------------
# Observed WBT27
obs.hus=get(load(obs_huss_file))
obs.Tx=get(load(obs_tasmax_file))
obs.hus=getTemporalIntersection(obs =obs.Tx,prd=obs.hus,which.return = 'prd')
obs.Tx=getTemporalIntersection(obs =obs.hus,prd=obs.Tx,which.return = 'prd')
obs_wbt <- calc_wbt(tas_grid = obs.Tx, hus_grid = obs.hus)

wbt27_obs <- aggregate_djf_Ind27(obs_wbt)
wbt27_obs_clim <- climatology(wbt27_obs)


#RCMs
rcm.Tx <-get(load(model_tasmax_file))
rcm.hus <-get(load(model_huss_file))
rcm.hus=getTemporalIntersection(obs =rcm.Tx,prd=rcm.hus,which.return = 'prd')
rcm.Tx=getTemporalIntersection(obs =rcm.hus,prd=rcm.Tx,which.return = 'prd')
rcm_wbt <- calc_wbt(tas_grid = rcm.Tx, hus_grid = rcm.hus)

# Raw model
wbt27_raw <- aggregate_djf_Ind27(rcm_wbt)

# Container for the bias maps
bias_wbt27 <- list()

bias_wbt27[["raw"]] <- gridArithmetics(
  climatology(wbt27_raw),
  wbt27_obs_clim,
  operator = "-"
)

[2026-09-10 10:30:42] Calculating TXth ...

[2026-09-10 10:30:44] Done

[2026-09-10 10:30:44] - Computing climatology...

[2026-09-10 10:30:45] - Done.

[2026-09-10 10:30:51] Calculating TXth ...

[2026-09-10 10:30:54] Done

[2026-09-10 10:30:54] - Computing climatology...

[2026-09-10 10:30:54] - Done.



In [28]:
for (method in methods_to_run) {
  # Indirect approach using multivariated methods with specific humidity
  wbt27_indirect_SH <- aggregate_djf_Ind27(
    wbt_indirect_SH[[method]]
  )

  bias_wbt27[[paste0("Indirect_SH_", toupper(method))]] <-
    gridArithmetics(
      climatology(wbt27_indirect_SH),
      wbt27_obs_clim,
      operator = "-"
    )

    }

[2026-09-10 10:35:41] Calculating TXth ...

[2026-09-10 10:35:44] Done

[2026-09-10 10:35:44] - Computing climatology...

[2026-09-10 10:35:44] - Done.



In [29]:
sessionInfo()

R version 3.6.3 (2020-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 8.10 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /nfs/home/gmeteo/balmacedar/micromamba/envs/R363-climate4R/lib/libopenblasp-r0.3.30.so

locale:
 [1] LC_CTYPE=en_GB.UTF-8          LC_NUMERIC=C                 
 [3] LC_TIME=en_GB.UTF-8           LC_COLLATE=en_GB.UTF-8       
 [5] LC_MONETARY=en_GB.UTF-8       LC_MESSAGES=en_GB.UTF-8      
 [7] LC_PAPER=en_GB.UTF-8          LC_NAME=en_GB.UTF-8          
 [9] LC_ADDRESS=en_GB.UTF-8        LC_TELEPHONE=en_GB.UTF-8     
[11] LC_MEASUREMENT=en_GB.UTF-8    LC_IDENTIFICATION=en_GB.UTF-8

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] climate4R.indices_0.3.1 magrittr_2.0.1          HeatStress_1.0.8.1     
 [4] MBC_0.10-8              FNN_1.1.3               energy_1.7-12          
 [7] Matrix_1.3-3            downscaleR_3.3.2        visualizeR_1.6.4       